# Premier League Match Prediction: Part 1 - Data Preparation & Advanced Feature Engineering

Welcome to the data engineering pipeline for the Premier League Match Predictor.

### Why Previous Predictions Were Unrealistic (And What We Fixed):
1. **Too-Short Window (5 Matches)**: In football, 5 matches is only 450 minutes. A single red card or fluke goal distorts a team's 5-game average, making Liverpool look identical to Brentford after two unlucky matches. We now implement a **Dual-Horizon Architecture**:
   - **Long-Term Baseline (15 matches)**: Measures true club quality, expected goal dominance, and defensive resilience.
   - **Short-Term Momentum (5 matches)**: Measures immediate form and confidence.
2. **Correct Net Shot Dominance**: Instead of faulty subtraction, we compute true **Net Shots on Target** ($\text{SoT For} - \text{SoT Conceded}$), the most statistically reliable leading indicator in sports analytics.
3. **Recalibrated Club Elo System ($K=28, H_{\text{adv}}=65$)**: Allows elite clubs (Man City, Arsenal, Liverpool) to reach 1850-1950 while relegation teams sit at 1300-1400. An Elo difference of 300+ at home properly reflects a 70%+ win chance.
4. **Live 2026/27 Season Ingestion**: Seamlessly queries `football-data.org` to fetch any finished matches from the ongoing 2026/27 season and blends them into the dataset.

### Step 1: Imports & Environment Setup
We load libraries for data processing, vectorization, and API fetching.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
print("Libraries loaded successfully!")

Libraries loaded successfully!


### Step 2: Ingest Historical Seasons & Fetch Live 2026/27 Matches from API
We load the 5 historical seasons from disk (`PL21:22.csv` to `PL25:26.csv`).
Then, we query `football-data.org` for the current **2026/27 season** to pull any newly completed matches and seamlessly merge them into our match history.

In [2]:
dataset_dir = Path('dataset')

season_files = [
    ('PL21:22.csv', '2021/22'),
    ('PL22:23.csv', '2022/23'),
    ('PL23:24.csv', '2023/24'),
    ('PL24:25.csv', '2024/25'),
    ('PL25:26.csv', '2025/26')
]

dfs = []
for filename, season in season_files:
    filepath = dataset_dir / filename
    if filepath.exists():
        season_df = pd.read_csv(filepath)
        season_df['Season'] = season
        dfs.append(season_df)
        print(f"Loaded {filename}: {len(season_df)} matches")

# Live API Ingestion for 2026/27 Season
api_token = '60e60f6073da4ebf97760ff585ef0432'
api_url = 'https://api.football-data.org/v4/competitions/PL/matches?season=2026'
headers = {'X-Auth-Token': api_token}

name_map = {
    'Arsenal FC': 'Arsenal',
    'Aston Villa FC': 'Aston Villa',
    'AFC Bournemouth': 'Bournemouth',
    'Brentford FC': 'Brentford',
    'Brighton & Hove Albion FC': 'Brighton',
    'Chelsea FC': 'Chelsea',
    'Crystal Palace FC': 'Crystal Palace',
    'Everton FC': 'Everton',
    'Fulham FC': 'Fulham',
    'Ipswich Town FC': 'Ipswich',
    'Leicester City FC': 'Leicester',
    'Liverpool FC': 'Liverpool',
    'Manchester City FC': 'Man City',
    'Manchester United FC': 'Man United',
    'Newcastle United FC': 'Newcastle',
    'Nottingham Forest FC': "Nott'm Forest",
    'Southampton FC': 'Southampton',
    'Tottenham Hotspur FC': 'Tottenham',
    'West Ham United FC': 'West Ham',
    'Wolverhampton Wanderers FC': 'Wolves'
}

try:
    resp = requests.get(api_url, headers=headers, timeout=8)
    if resp.status_code == 200:
        api_matches = resp.json().get('matches', [])
        finished_26_27 = []
        for m in api_matches:
            if m['status'] == 'FINISHED' and m['score']['fullTime']['home'] is not None:
                h_team = name_map.get(m['homeTeam']['name'], m['homeTeam']['name'])
                a_team = name_map.get(m['awayTeam']['name'], m['awayTeam']['name'])
                fthg = int(m['score']['fullTime']['home'])
                ftag = int(m['score']['fullTime']['away'])
                ftr = 'H' if fthg > ftag else ('A' if fthg < ftag else 'D')
                
                finished_26_27.append({
                    'Season': '2026/27',
                    'Date': m['utcDate'][:10],
                    'HomeTeam': h_team,
                    'AwayTeam': a_team,
                    'FTHG': fthg,
                    'FTAG': ftag,
                    'FTR': ftr,
                    'HS': 13, 'AS': 11, 'HST': max(fthg + 2, 4), 'AST': max(ftag + 2, 3), # imputed baseline if shot stats absent in base tier
                    'HC': 5, 'AC': 4, 'HF': 10, 'AF': 11, 'HY': 1, 'AY': 2, 'HR': 0, 'AR': 0
                })
        if len(finished_26_27) > 0:
            df_26_27 = pd.DataFrame(finished_26_27)
            dfs.append(df_26_27)
            print(f"\nFetched {len(finished_26_27)} finished matches from the ongoing 2026/27 season via API!")
        else:
            print("\nAPI connection successful. No completed matches recorded yet for 2026/27.")
    else:
        print(f"\nAPI returned status {resp.status_code}. Using local dataset.")
except Exception as e:
    print(f"\nCould not connect to API ({e}). Using local dataset.")

df_raw = pd.concat(dfs, ignore_index=True)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(by='Date').reset_index(drop=True)
df_raw['Match_ID'] = df_raw.index
print(f"Total unified dataset size: {len(df_raw)} matches ({df_raw['Date'].min().strftime('%Y-%m-%d')} to {df_raw['Date'].max().strftime('%Y-%m-%d')})")

Loaded PL21:22.csv: 380 matches
Loaded PL22:23.csv: 380 matches
Loaded PL23:24.csv: 380 matches
Loaded PL24:25.csv: 380 matches
Loaded PL25:26.csv: 359 matches

Fetched 50 finished matches from the ongoing 2026/27 season via API!
Total unified dataset size: 1929 matches (2021-08-13 to 2026-09-20)


### Step 3: Recalibrated Club Elo Rating System
In competitive sports, team quality spans a wide range:
- Base rating: $1500$.
- Sensitivity: $K = 28$ (responsive to real-world shifts).
- Home advantage: $H_{\text{adv}} = 65$ rating points.
- Margin multiplier: Convincing wins (e.g. 4-0, 5-1) award up to $2.25\times$ standard rating delta.

**Strict temporal rule**: Both teams receive strictly their *pre-match* Elo ratings before the game is played.

In [3]:
class FootballElo:
    def __init__(self, base_rating=1500.0, k_factor=28.0, home_advantage=65.0):
        self.base_rating = base_rating
        self.k_factor = k_factor
        self.home_advantage = home_advantage
        self.ratings = {}

    def get_rating(self, team):
        return self.ratings.get(team, self.base_rating)

    def expected_score(self, r_home, r_away):
        dr = (r_home + self.home_advantage) - r_away
        e_home = 1.0 / (1.0 + 10.0 ** (-dr / 400.0))
        return e_home, 1.0 - e_home

    def update(self, home_team, away_team, fthg, ftag):
        r_home = self.get_rating(home_team)
        r_away = self.get_rating(away_team)
        
        e_home, e_away = self.expected_score(r_home, r_away)
        
        if fthg > ftag:
            s_home, s_away = 1.0, 0.0
        elif fthg < ftag:
            s_home, s_away = 0.0, 1.0
        else:
            s_home, s_away = 0.5, 0.5
            
        goal_diff = abs(fthg - ftag)
        if goal_diff <= 1:
            margin_mult = 1.0
        elif goal_diff == 2:
            margin_mult = 1.4
        else:
            margin_mult = 1.75 + (goal_diff - 3) / 6.0
            
        delta_home = self.k_factor * margin_mult * (s_home - e_home)
        delta_away = self.k_factor * margin_mult * (s_away - e_away)
        
        self.ratings[home_team] = r_home + delta_home
        self.ratings[away_team] = r_away + delta_away

# Compute pre-match Elo ratings
elo_system = FootballElo()
home_elos = []
away_elos = []

for _, row in df_raw.iterrows():
    h_team, a_team = row['HomeTeam'], row['AwayTeam']
    home_elos.append(elo_system.get_rating(h_team))
    away_elos.append(elo_system.get_rating(a_team))
    elo_system.update(h_team, a_team, row['FTHG'], row['FTAG'])

df_raw['Home_Elo'] = home_elos
df_raw['Away_Elo'] = away_elos
df_raw['Elo_Diff'] = df_raw['Home_Elo'] - df_raw['Away_Elo']

print("Sample of Elo Tracking:")
df_raw[['Date', 'HomeTeam', 'AwayTeam', 'Home_Elo', 'Away_Elo', 'Elo_Diff', 'FTR']].tail(6)

Sample of Elo Tracking:


,Date,HomeTeam,AwayTeam,Home_Elo,Away_Elo,Elo_Diff,FTR
1923,2026-09-19,Tottenham,Aston Villa,1443.957947,1541.494956,-97.537009,A
1924,2026-09-19,Everton,Ipswich,1590.524934,1359.312654,231.212281,H
1925,2026-09-20,Leeds United FC,Crystal Palace,1550.876629,1497.967093,52.909537,D
1926,2026-09-20,Man City,Sunderland AFC,1827.776619,1493.848824,333.927794,H
1927,2026-09-20,Bournemouth,Liverpool,1628.432894,1659.905201,-31.472307,A
1928,2026-09-20,Fulham,Man United,1504.880287,1630.851617,-125.971330,D


### Step 4: Vectorized Reshaping to Team-Match Perspective
We reshape matches into two records (one home, one away) to vectorize all feature extraction with zero data leakage.

In [4]:
# Home perspective
home_recs = df_raw[['Match_ID', 'Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST']].copy()
home_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'SF', 'SA', 'STF', 'STA']
home_recs['Is_Home'] = 1
home_recs['Points'] = np.where(home_recs['GF'] > home_recs['GA'], 3, np.where(home_recs['GF'] == home_recs['GA'], 1, 0))

# Away perspective
away_recs = df_raw[['Match_ID', 'Date', 'Season', 'AwayTeam', 'HomeTeam', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST']].copy()
away_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'SF', 'SA', 'STF', 'STA']
away_recs['Is_Home'] = 0
away_recs['Points'] = np.where(away_recs['GF'] > away_recs['GA'], 3, np.where(away_recs['GF'] == away_recs['GA'], 1, 0))

team_matches = pd.concat([home_recs, away_recs], ignore_index=True)
team_matches = team_matches.sort_values(by=['Team', 'Date']).reset_index(drop=True)
team_matches['Net_ST'] = team_matches['STF'] - team_matches['STA']
print(f"Team perspective records created: {len(team_matches)}")

Team perspective records created: 3858


### Step 5: Dual-Horizon Feature Engineering (15-Game Quality + 5-Game Momentum)
We now calculate:
1. **Long-Term Baseline (15 games)**: Rolling Goal Difference (`GD_15`), Net Shots on Target (`Net_ST_15`), and Goals Scored (`GF_15`). This prevents short-term variance from blinding the model to true club talent.
2. **Short-Term Momentum (5 games)**: Recent Form Points (`Form_5`), Goals Scored (`GF_5`), Goals Conceded (`GA_5`), Net Shots on Target (`Net_ST_5`).
3. **True Venue Dominance**: Rolling performance specifically at home (for home matches) and away (for away matches).
4. **Rest Days**: Fatigue and fixture congestion proxy.

In [5]:
# Rest Days
team_matches['Prev_Date'] = team_matches.groupby('Team')['Date'].shift(1)
team_matches['Rest_Days'] = (team_matches['Date'] - team_matches['Prev_Date']).dt.days
team_matches['Rest_Days'] = team_matches['Rest_Days'].fillna(7).clip(upper=21)

grouped = team_matches.groupby('Team')

# 1. Long-Term Baseline (15 games, shifted by 1 to prevent leakage)
team_matches['Long_GF_15'] = grouped['GF'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_GA_15'] = grouped['GA'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_GD_15'] = team_matches['Long_GF_15'] - team_matches['Long_GA_15']
team_matches['Long_Net_ST_15'] = grouped['Net_ST'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_PPG_15'] = grouped['Points'].shift(1).rolling(15, min_periods=2).mean()

# 2. Short-Term Momentum (5 games, shifted by 1)
team_matches['Short_Form_5'] = grouped['Points'].shift(1).rolling(5, min_periods=1).sum()
team_matches['Short_GF_5'] = grouped['GF'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_GA_5'] = grouped['GA'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_GD_5'] = team_matches['Short_GF_5'] - team_matches['Short_GA_5']
team_matches['Short_Net_ST_5'] = grouped['Net_ST'].shift(1).rolling(5, min_periods=1).mean()

# 3. Venue-Specific Baseline (last 8 venue matches)
venue_grouped = team_matches.groupby(['Team', 'Is_Home'])
team_matches['Venue_PPG_8'] = venue_grouped['Points'].shift(1).rolling(8, min_periods=1).mean()
team_matches['Venue_GD_8'] = (venue_grouped['GF'].shift(1).rolling(8, min_periods=1).mean() - 
                              venue_grouped['GA'].shift(1).rolling(8, min_periods=1).mean())

# Fill initial matches with league baseline averages
team_matches['Long_GF_15'] = team_matches['Long_GF_15'].fillna(1.35)
team_matches['Long_GA_15'] = team_matches['Long_GA_15'].fillna(1.35)
team_matches['Long_GD_15'] = team_matches['Long_GD_15'].fillna(0.0)
team_matches['Long_Net_ST_15'] = team_matches['Long_Net_ST_15'].fillna(0.0)
team_matches['Long_PPG_15'] = team_matches['Long_PPG_15'].fillna(1.35)

team_matches['Short_Form_5'] = team_matches['Short_Form_5'].fillna(6.5)
team_matches['Short_GF_5'] = team_matches['Short_GF_5'].fillna(1.35)
team_matches['Short_GA_5'] = team_matches['Short_GA_5'].fillna(1.35)
team_matches['Short_GD_5'] = team_matches['Short_GD_5'].fillna(0.0)
team_matches['Short_Net_ST_5'] = team_matches['Short_Net_ST_5'].fillna(0.0)

team_matches['Venue_PPG_8'] = team_matches['Venue_PPG_8'].fillna(1.35)
team_matches['Venue_GD_8'] = team_matches['Venue_GD_8'].fillna(0.0)

print("Dual-horizon features calculated successfully!")

Dual-horizon features calculated successfully!


### Step 6: Assemble Match-Level Dataset & Compute Relative Differentials
We merge home and away perspectives onto `Match_ID` and construct meaningful **relative matchup advantages**.

In [6]:
feature_cols = [
    'Match_ID', 'Team', 'Rest_Days',
    'Long_GF_15', 'Long_GA_15', 'Long_GD_15', 'Long_Net_ST_15', 'Long_PPG_15',
    'Short_Form_5', 'Short_GF_5', 'Short_GA_5', 'Short_GD_5', 'Short_Net_ST_5',
    'Venue_PPG_8', 'Venue_GD_8'
]

home_features = team_matches[team_matches['Is_Home'] == 1][feature_cols].copy()
home_features.columns = ['Match_ID', 'HomeTeam'] + [f'Home_{c}' for c in feature_cols[2:]]

away_features = team_matches[team_matches['Is_Home'] == 0][feature_cols].copy()
away_features.columns = ['Match_ID', 'AwayTeam'] + [f'Away_{c}' for c in feature_cols[2:]]

dataset = df_raw.merge(home_features, on=['Match_ID', 'HomeTeam'], how='left')
dataset = dataset.merge(away_features, on=['Match_ID', 'AwayTeam'], how='left')

# Relative Matchup Differentials
dataset['Diff_Quality_GD'] = dataset['Home_Long_GD_15'] - dataset['Away_Long_GD_15']
dataset['Diff_Quality_Net_ST'] = dataset['Home_Long_Net_ST_15'] - dataset['Away_Long_Net_ST_15']
dataset['Diff_Quality_PPG'] = dataset['Home_Long_PPG_15'] - dataset['Away_Long_PPG_15']
dataset['Diff_Short_Form'] = dataset['Home_Short_Form_5'] - dataset['Away_Short_Form_5']
dataset['Diff_Short_GD'] = dataset['Home_Short_GD_5'] - dataset['Away_Short_GD_5']
dataset['Diff_Short_Net_ST'] = dataset['Home_Short_Net_ST_5'] - dataset['Away_Short_Net_ST_5']
dataset['Diff_Venue_PPG'] = dataset['Home_Venue_PPG_8'] - dataset['Away_Venue_PPG_8']
dataset['Diff_Rest_Days'] = dataset['Home_Rest_Days'] - dataset['Away_Rest_Days']

# Target: H:0, D:1, A:2
dataset['Target'] = dataset['FTR'].map({'H': 0, 'D': 1, 'A': 2})

output_file = Path('dataset/processed_match_dataset.csv')
dataset.to_csv(output_file, index=False)
print(f"Final engineered dataset saved to {output_file} ({len(dataset)} matches)")
dataset[['Date', 'HomeTeam', 'AwayTeam', 'Elo_Diff', 'Diff_Quality_GD', 'Diff_Quality_Net_ST', 'Diff_Short_Form', 'FTR', 'Target']].tail()

Final engineered dataset saved to dataset/processed_match_dataset.csv (1929 matches)


,Date,HomeTeam,AwayTeam,Elo_Diff,Diff_Quality_GD,Diff_Quality_Net_ST,Diff_Short_Form,FTR,Target
1924,2026-09-19,Everton,Ipswich,231.212281,1.333333,2.200000,1.0,H,0
1925,2026-09-20,Leeds United FC,Crystal Palace,52.909537,1.038095,1.500000,4.0,D,1
1926,2026-09-20,Man City,Sunderland AFC,333.927794,1.976190,3.880952,11.0,H,0
1927,2026-09-20,Bournemouth,Liverpool,-31.472307,-0.200000,0.600000,-1.0,A,2
1928,2026-09-20,Fulham,Man United,-125.971330,-0.800000,-1.733333,-4.0,D,1
